Drive + paths

In [1]:
import os
os.makedirs("outputs", exist_ok=True)

TRAIN_CSV = "train.csv"
DEV_CSV   = "dev.csv"
OUT = "outputs"
print("Using local Colab storage")

Using local Colab storage


Install deps

In [ ]:
!pip -q install tensorflow pandas numpy scikit-learn gensim tqdm

Imports + config

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (Input, Embedding, Bidirectional, LSTM, Dense, Dropout,
                                     Concatenate, Multiply, Subtract, Dot, Softmax,
                                     GlobalAveragePooling1D)
from tensorflow.keras.models import Model
import pickle

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

VOCAB_SIZE = 30000
EMBED_DIM  = 300
MAX_LEN    = 60     # we’ll set this automatically next cell if you want
BATCH_SIZE = 64
EPOCHS     = 15

: 

Load train/dev + set MAX_LEN from 95th percentile (recommended)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(DEV_CSV)

assert {"premise","hypothesis","label"}.issubset(train_df.columns)
assert {"premise","hypothesis","label"}.issubset(dev_df.columns)

prem_lens = train_df["premise"].astype(str).str.split().str.len().values
hyp_lens  = train_df["hypothesis"].astype(str).str.split().str.len().values
p95 = int(np.percentile(np.concatenate([prem_lens, hyp_lens]), 95))
MAX_LEN = int(np.ceil(p95 / 5) * 5)

print("train:", train_df.shape, "dev:", dev_df.shape)
print("Using MAX_LEN:", MAX_LEN)
print("train label dist:", train_df["label"].value_counts(normalize=True).to_dict())

train: (24432, 3) dev: (6736, 3)
Using MAX_LEN: 35
train label dist: {1: 0.5176817288801572, 0: 0.4823182711198428}


Tokenize + pad

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts((train_df.premise.astype(str) + " " + train_df.hypothesis.astype(str)).tolist())

def tok_pad(series):
    seqs = tokenizer.texts_to_sequences(series.astype(str).tolist())
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

X_train_p = tok_pad(train_df.premise)
X_train_h = tok_pad(train_df.hypothesis)
y_train   = train_df.label.values.astype(int)

X_dev_p = tok_pad(dev_df.premise)
X_dev_h = tok_pad(dev_df.hypothesis)
y_dev   = dev_df.label.values.astype(int)

print(X_train_p.shape, X_train_h.shape, y_train.shape)

(24432, 35) (24432, 35) (24432,)


Save tokenizer (for demo notebook later)

In [ ]:
tok_path = f"{OUT}/tokenizer.pkl"
with open(tok_path, "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Saved tokenizer:", tok_path)

✅ Saved tokenizer: outputs/tokenizer.pkl


Load FastText vectors (gensim downloader)

In [ ]:
import gensim.downloader as api

# FastText 300d with subwords (big download)
ft = api.load("fasttext-wiki-news-subwords-300")
print("✅ FastText loaded")

✅ FastText loaded


Build embedding matrix

In [ ]:
word_index = tokenizer.word_index
num_words = min(VOCAB_SIZE, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, EMBED_DIM), dtype=np.float32)

hits = 0
for w, i in word_index.items():
    if i >= num_words:
        continue
    if w in ft:
        embedding_matrix[i] = ft[w]
        hits += 1

print("num_words:", num_words)
print("hit-rate:", hits / max(1, num_words))

num_words: 30000
hit-rate: 0.8282333333333334


Build Decomposable Attention model (minimal Keras)

In [ ]:
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Dropout,
    Concatenate, Multiply, Subtract, Dot, Softmax,
    GlobalAveragePooling1D, Lambda
)
from tensorflow.keras.models import Model
import tensorflow as tf

def build_model():
    prem_in = Input(shape=(MAX_LEN,), name="premise")
    hyp_in  = Input(shape=(MAX_LEN,), name="hypothesis")

    emb = Embedding(
        input_dim=num_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=False,
        mask_zero=False,   # ✅ disable masking to avoid BroadcastTo error
        name="fasttext_emb"
    )

    prem = emb(prem_in)
    hyp  = emb(hyp_in)

    enc = Bidirectional(LSTM(200, return_sequences=True), name="bilstm")
    A = enc(prem)
    B = enc(hyp)

    sim = Dot(axes=-1)([A, B])  # [B,L,L]

    alpha = Softmax(axis=-1)(sim)
    beta  = Softmax(axis=1)(sim)

    A_tilde = Lambda(lambda x: tf.matmul(x[0], x[1]))([alpha, B])
    beta_T  = Lambda(lambda x: tf.transpose(x, [0,2,1]))(beta)
    B_tilde = Lambda(lambda x: tf.matmul(x[0], x[1]))([beta_T, A])

    A_feat = Concatenate()([A, A_tilde, Subtract()([A, A_tilde]), Multiply()([A, A_tilde])])
    B_feat = Concatenate()([B, B_tilde, Subtract()([B, B_tilde]), Multiply()([B, B_tilde])])

    comp = Dense(200, activation="relu")
    A_comp = comp(A_feat)
    B_comp = comp(B_feat)

    A_vec = GlobalAveragePooling1D()(A_comp)
    B_vec = GlobalAveragePooling1D()(B_comp)

    x = Concatenate()([A_vec, B_vec])
    x = Dense(200, activation="relu")(x)
    x = Dropout(0.35)(x)
    out = Dense(1, activation="sigmoid")(x)

    return Model([prem_in, hyp_in], out)

model = build_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ premise             │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hypothesis          │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fasttext_emb        │ (None, 35, 300)   │  9,000,000 │ premise[0][0],    │
│ (Embedding)         │                   │            │ hypothesis[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, 35, 400)   │    801,600 │ fasttext_emb[0][… │
│ (Bidirectional)     │                   │            │ fasttext_emb[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 35, 35)    │          0 │ bilstm[0][0],     │
│                     │                   │            │ bilstm[1][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_3 (Softmax) │ (None, 35, 35)    │          0 │ dot_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_2 (Softmax) │ (None, 35, 35)    │          0 │ dot_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_4 (Lambda)   │ (None, 35, 35)    │          0 │ softmax_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 35, 400)   │          0 │ softmax_2[0][0],  │
│                     │                   │            │ bilstm[1][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_5 (Lambda)   │ (None, 35, 400)   │          0 │ lambda_4[0][0],   │
│                     │                   │            │ bilstm[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_2          │ (None, 35, 400)   │          0 │ bilstm[0][0],     │
│ (Subtract)          │                   │            │ lambda_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 35, 400)   │          0 │ bilstm[0][0],     │
│ (Multiply)          │                   │            │ lambda_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_3          │ (None, 35, 400)   │          0 │ bilstm[1][0],     │
│ (Subtract)          │                   │            │ lambda_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 35, 400)   │          0 │ bilstm[1][0],     │
│ (Multiply)          │                   │            │ lambda_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 35, 1600)  │          0 │ bilstm[0][0],     │
│ (Concatenate)       │                   │            │ lambda_3[0][0],   │
│                     │                   │            │ subtract_2[0][0], │
│                     │                   │            │ multiply_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 35, 1600)  │          0 │ bilstm[1][0],     │
│ (Concatenate)       │                   │            │ lambda_5[0][0],   │
│                     │                   │            │ subtract_3[0][0], │
│                     │                   │            │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 10,202,201 (38.92 MB)

 Trainable params: 1,202,201 (4.59 MB)

 Non-trainable params: 9,000,000 (34.33 MB)

Compile + callbacks

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(f"{OUT}/best_model.keras", monitor="val_loss", save_best_only=True)
]

Train

In [ ]:
history = model.fit(
    x=[X_train_p, X_train_h],
    y=y_train,
    validation_data=([X_dev_p, X_dev_h], y_dev),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/15
382/382 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7158 - loss: 0.5328 - val_accuracy: 0.6882 - val_loss: 0.5825
Epoch 2/15
382/382 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.7231 - loss: 0.5205 - val_accuracy: 0.6872 - val_loss: 0.5873
Epoch 3/15
382/382 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7376 - loss: 0.5080 - val_accuracy: 0.6899 - val_loss: 0.6010
Epoch 4/15
382/382 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7477 - loss: 0.4928 - val_accuracy: 0.6918 - val_loss: 0.6055


Dev evaluation (F1 + report)

In [ ]:
dev_probs = model.predict([X_dev_p, X_dev_h], batch_size=256).ravel()
dev_pred  = (dev_probs >= 0.5).astype(int)

print("Dev F1:", f1_score(y_dev, dev_pred))
print(classification_report(y_dev, dev_pred, digits=4))

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
Dev F1: 0.7056502747639848
              precision    recall  f1-score   support

           0     0.6875    0.6578    0.6723      3258
           1     0.6919    0.7200    0.7057      3478

    accuracy                         0.6899      6736
   macro avg     0.6897    0.6889    0.6890      6736
weighted avg     0.6898    0.6899    0.6895      6736



Save small config (helps write-up)

In [ ]:
import json
cfg = {
    "VOCAB_SIZE": VOCAB_SIZE,
    "EMBED_DIM": EMBED_DIM,
    "MAX_LEN": MAX_LEN,
    "encoder": "BiLSTM(200)",
    "embedding": "fasttext-wiki-news-subwords-300",
    "embedding_trainable": False
}
with open(f"{OUT}/config.json","w") as f:
    json.dump(cfg, f, indent=2)
print("✅ Saved config:", f"{OUT}/config.json")
print("✅ Best model:", f"{OUT}/best_model.keras")

✅ Saved config: outputs/config.json
✅ Best model: outputs/best_model.keras
